# <font color="#003660">Applied Machine Learning for Text Analysis (M.184.5331)</font>

# <font color="#003660">Session 5: Retrieval-Augmented Generation</font>

# <font color="#003660">Reranking</font>

<center><br><img width=256 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/dag.png"/><br></center>

<p>

<div>
    <font color="#085986"><b>By the end of this lesson, you ...</b><br><br>
        ... know how to implement reranking methods with LangChain, <br>
        ... know how to implement LangChain chains on your own.
    </font>
</div>
</p>

Inspired by these excellent sources:

* [HuggingFace (2024): NLP Course](https://huggingface.co/learn/nlp-course/)
* [Huggingface (2024): Open-Source AI Cookbook](https://huggingface.co/learn/cookbook/index)
* [LangChain Docs: Build a RAG agent](https://docs.langchain.com/oss/python/langchain/rag)
* [LangChain API Reference](https://reference.langchain.com/python/)
* [LangChain AI (2024) Cookbook](https://github.com/langchain-ai/langchain/blob/master/cookbook/rewrite.ipynb?ref=blog.langchain.dev)

## Where We Are

* **5.03:** RAG with LangChain. The retriever returns the 3 chunks that are most similar to the question.
* **Now:** **reranking**. Retrieve more chunks, then let a second model pick the best ones.

![RAG extensions](https://raw.githubusercontent.com/olivermueller/amlta-2026/main/Session_05/imgs/rag_extensions.png)

(Source: [Wang et al., 2024](https://doi.org/10.18653/v1/2024.emnlp-main.981))

Reranking is one of many ways to improve RAG ([Wang et al., 2024](https://doi.org/10.18653/v1/2024.emnlp-main.981)).

In [ ]:
%pip install -q langchain langchain-classic langchain-openai langchain-community langchain-text-splitters faiss-cpu sentence-transformers flashrank python-dotenv

In [ ]:
import os
import re
import urllib.request

from IPython.display import display, Markdown

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_community.document_compressors import FlashrankRerank

In [ ]:
# downloads the documents, if they are not here yet
BASE_URL = "https://raw.githubusercontent.com/olivermueller/amlta-2026/main/Session_05/"
for path in [
    "documents/Game_of_Thrones.pdf",
    "documents/How_I_Met_Your_Mother.pdf",
    "markdown_documents/Game_of_Thrones.md",
    "markdown_documents/How_I_Met_Your_Mother.md",
]:
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        urllib.request.urlretrieve(BASE_URL + path, path)

### Connect to the AI-Gateway

The same setup as in 4.03: your own computer, connected to the university network or the VPN, and your AI-Gateway key.

We use two models on the gateway:

* a **chat model** that writes the answers (the generator), and
* an **embedding model** that turns texts into vectors (the retriever).

**Good to know:** the gateway allows about 10 requests per minute per model. If a cell pauses, the client is waiting for the next minute and then continues by itself.

In [ ]:
from dotenv import load_dotenv

load_dotenv()  # reads the .env file: searches this folder first, then the folders above (the repo folder)
API_KEY = os.getenv("AI_GATEWAY_KEY")
AI_GATEWAY_BASE = os.getenv("AI_GATEWAY_BASE")  # address of the university's AI-Gateway
if not API_KEY or not AI_GATEWAY_BASE:
    raise ValueError("AI_GATEWAY_KEY or AI_GATEWAY_BASE missing. Copy .env.example to .env in the repo folder and paste your key there.")

GENERATOR_NAME = "gwdg.qwen3.8-27b-no-think"  # chat model: writes the answers
RETRIEVER_NAME = "gwdg.qwen3-embedding-4b"    # embedding model: turns texts into vectors

The key comes from the `.env` file in the repo folder (set up in 4.03). If the cell says that no key was found, copy `.env.example` to `.env` and paste your key behind `AI_GATEWAY_KEY=`.

## Loading the Documents

The Markdown files from 5.02, with the links removed.

In [ ]:
markdown_documents_path = "markdown_documents"

In [ ]:
def remove_markdown_links(text: str) -> str:
    # 1. Remove inline links: [text](url)
    text = re.sub(r'\[([^\]]*)\]\([^)]+\)', r'\1', text)

    # 2. Remove reference-style links: [text][ref]
    text = re.sub(r'\[([^\]]*)\]\[[^\]]*\]', r'\1', text)

    # 3. Remove malformed/broken bracket groups like [[][1][]]
    text = re.sub(r'\[+\s*\]+', '', text)

    # 4. Remove reference definitions like: [1]: https://...
    text = re.sub(r'^\s*\[[^\]]+\]:.*$', '', text, flags=re.MULTILINE)

    return text

In [ ]:
markdown_documents = sorted(os.listdir(markdown_documents_path))

md_files = []

for markdown_document in markdown_documents:
    markdown_document_path = os.path.join(markdown_documents_path, markdown_document)
    with open(markdown_document_path, encoding="utf-8") as file:
        md_files.append([markdown_document, remove_markdown_links(file.read())])

## The Chain from 5.03

### The vector store: build it once, then load it

Embedding all chunks takes about a minute and costs about 150,000 tokens of your gateway budget. So we do it only once:

* If you ran **5.03** in this folder, the index is saved in the folder `faiss_index`, and we just load it.
* Otherwise, we build it now and save it for the next notebooks.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
all_splits = text_splitter.create_documents(
    texts=[x[1] for x in md_files],
    metadatas=[{"source": x[0]} for x in md_files],
)

retriever_model = OpenAIEmbeddings(
    model=RETRIEVER_NAME,
    base_url=AI_GATEWAY_BASE,
    api_key=API_KEY,
    check_embedding_ctx_length=False,  # send plain text; the default would first split it with OpenAI's own tokenizer
)

if os.path.exists("faiss_index"):
    db = FAISS.load_local("faiss_index", retriever_model, allow_dangerous_deserialization=True)  # our own file, so loading is safe
else:
    db = FAISS.from_documents(all_splits, embedding=retriever_model)
    db.save_local("faiss_index")
print(db.index.ntotal, "chunks in the vector store")

For reranking, the retriever returns the **10** most similar chunks:

* **Without reranking**, we simply use the first 3 of them.
* **With reranking**, a second model reads all 10 again and picks the best 3.

In [ ]:
retriever = db.as_retriever(search_kwargs={"k": 10})

In [ ]:
llm = ChatOpenAI(model=GENERATOR_NAME, base_url=AI_GATEWAY_BASE, api_key=API_KEY, temperature=0)

In [ ]:
prompt_template = """
You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:
"""
prompt = PromptTemplate.from_template(prompt_template)
print(prompt.template)

## Part 1: The Cross-Encoder Reranker

**In short: a second model reads the question and each chunk together, and sorts the chunks again.**

![](https://raw.githubusercontent.com/UKPLab/sentence-transformers/master/docs/img/Bi_vs_Cross-Encoder.png)

Source: ([SBERT.net (2024)](https://www.sbert.net/examples/applications/cross-encoder/README.html))

| | Bi-encoder (our embedding model) | Cross-encoder (the reranker) |
|---|---|---|
| Reads | question and chunk **separately** | question and chunk **together** |
| Returns | one vector per text | one score between 0 and 1 per pair |
| Speed | fast: the chunk vectors are computed once | slow: one model run per pair |
| Good for | searching millions of chunks | sorting the 10 retrieved chunks again |

([Reimers and Gurevych, 2019](https://doi.org/10.48550/arXiv.1908.10084))

The rerankers in this notebook are small models that run on your own computer. The first run downloads them.

In [ ]:
model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")
compressor = CrossEncoderReranker(model=model, top_n=3)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=retriever
)

Which chunks come first, without and with reranking? The numbers are the ids of the chunks:

In [ ]:
print([x.id for x in retriever.invoke("Who plays Daenerys Targaryen?")[:3]])
print([x.id for x in compression_retriever.invoke("Who plays Daenerys Targaryen?")])

### Your turn

Complete the `"compression"` branch in `invoke_reranking_rag_chain()`: build the `ContextualCompressionRetriever` from above, and use it to retrieve the chunks.

Then run both cells below and compare: which chunks does the model get, and does the answer change?

In [ ]:
def invoke_reranking_rag_chain(question, retriever_type="basic"):
    if retriever_type == "basic":
        retrieved_docs = retriever.invoke(question)[:3]  # the 3 most similar chunks, without reranking
    elif retriever_type == "compression":
        # TODO: implement compression retriever in here
        pass

    elif retriever_type == "flash":
        # TODO: Implement Flash retriever

        # your code here
        pass

    else:
        raise NotImplementedError("No other retrievers implemented yet.")
    for doc in retrieved_docs:
        display(Markdown(f"### {doc.metadata['source']}"))
        display(Markdown(doc.page_content))
    print("#" * 50)
    input_prompt = prompt.invoke({"question": question, "context": "\n\n".join(doc.page_content for doc in retrieved_docs)})
    answer = llm.invoke(input_prompt).content
    return answer

In [ ]:
question = "Who plays Daenerys Targaryen?"
answer = invoke_reranking_rag_chain(question, "basic")
print("Answer:", answer)

In [ ]:
question = "Who plays Daenerys Targaryen?"
answer = invoke_reranking_rag_chain(question, "compression")
print("Answer:", answer)

## Part 2: FlashRank

**In short: the same idea with a smaller and faster reranker.**

[FlashRank](https://github.com/PrithivirajDamodaran/FlashRank) uses tiny cross-encoders that run fast on a normal CPU. LangChain wraps it as `FlashrankRerank`.

### Your turn

1. Build a `FlashrankRerank(top_n=3)` compressor and a `ContextualCompressionRetriever` with it, like for the cross-encoder.
2. Add it as the `"flash"` branch of the function below.
3. Compare the chunks and the answer with the other two retrievers.

In [ ]:
# TODO: Implement Flash retriever

# your code here

In [ ]:
def invoke_reranking_rag_chain(question, retriever_type="basic"):
    if retriever_type == "basic":
        retrieved_docs = retriever.invoke(question)[:3]  # the 3 most similar chunks, without reranking
    elif retriever_type == "compression":
        # TODO: implement compression retriever in here
        pass
    elif retriever_type == "flash":
        # TODO: Implement Flash retriever

        # your code here
        pass

    else:
        raise NotImplementedError("No other retrievers implemented yet.")
    for doc in retrieved_docs:
        display(Markdown(f"### {doc.metadata['source']}"))
        display(Markdown(doc.page_content))
    print("#" * 50)
    input_prompt = prompt.invoke({"question": question, "context": "\n\n".join(doc.page_content for doc in retrieved_docs)})
    answer = llm.invoke(input_prompt).content
    return answer

In [ ]:
question = "Who plays Daenerys Targaryen?"
answer = invoke_reranking_rag_chain(question, "flash")
print("Answer:", answer)

## What You Learned

* The retriever returns more chunks than we need (here 10). A **reranker** sorts them again and keeps the best 3.
* A **cross-encoder** reads the question and the chunk together. That makes it more precise than the embedding model, but too slow for searching everything.
* In LangChain, a `ContextualCompressionRetriever` puts the reranker behind the normal retriever.

Next: **5.05 RAG Query Rewriting**. A better question before the search.